In [1]:
%mamba install scikit-learn numpy pandas

mambajs 0.19.13

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, scikit-learn, pandas
Channels: emscripten-forge, conda-forge

Solving environment...
Solving took 2.9355 seconds
  Name                          Version                       Build                         Channel                       
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
+ brotli-python                 1.2.0                         py313h33caa6c_0               emscripten-forge              
+ certifi                       2026.4.22                     pyhd8ed1ab_0                  conda-forge                   
+ charset-normalizer            3.4.7                         pyhd8ed1ab_0                  conda-forge                   
+ idna                          3.13                          pyhcf101f3_0                  conda-forge                   
+ joblib                        1.5.3            

## Robust Scaling
Formula: `Xnew = (X - Q50) / IQR` where `IQR = Q75 - Q25`

In [2]:
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd

In [3]:
dataset = pd.DataFrame(
    {
        "Age":[30,20,40,50,33,23,34,60,45,32],
        "SemNo":[1,5,6,2,3,7,8,9,3,2],
        "Attendance":[90,34,89,70,60,50,78,88,99,100]
    }
)

In [4]:
# Method 1 — Manual Robust Scaling
dataset_scaled = pd.DataFrame()

for feature in dataset.columns:
    feature_set = np.array(dataset[feature])
    q25, q50, q75 = np.percentile(feature_set, [25, 50, 75])
    iqr = q75 - q25
    print(q25, q50, q75)
    feature_set = (feature_set - q50) / iqr
    dataset_scaled[feature] = feature_set

print('=============Without SK-Learn================')
print(dataset_scaled)

30.5 33.5 43.75
2.25 4.0 6.75
62.5 83.0 89.75
=============Without SK-Learn================
        Age     SemNo  Attendance
0 -0.264151 -0.666667    0.256881
1 -1.018868  0.222222   -1.798165
2  0.490566  0.444444    0.220183
3  1.245283 -0.444444   -0.477064
4 -0.037736 -0.222222   -0.844037
5 -0.792453  0.666667   -1.211009
6  0.037736  0.888889   -0.183486
7  2.000000  1.111111    0.183486
8  0.867925 -0.222222    0.587156
9 -0.113208 -0.444444    0.623853


In [5]:
# Method 2 — Using sklearn RobustScaler
print('=============SK-learn RobustScaler================')
sScaler = RobustScaler()
dataset_scaled1 = sScaler.fit_transform(dataset)
dataset_scaled1 = pd.DataFrame(dataset_scaled1, columns=dataset.columns)
print(dataset_scaled1)

=============SK-learn RobustScaler================
        Age     SemNo  Attendance
0 -0.264151 -0.666667    0.256881
1 -1.018868  0.222222   -1.798165
2  0.490566  0.444444    0.220183
3  1.245283 -0.444444   -0.477064
4 -0.037736 -0.222222   -0.844037
5 -0.792453  0.666667   -1.211009
6  0.037736  0.888889   -0.183486
7  2.000000  1.111111    0.183486
8  0.867925 -0.222222    0.587156
9 -0.113208 -0.444444    0.623853


In [6]:
# Cross-check: both methods should match
difference = pd.DataFrame()
for feature in dataset.columns:
    difference[feature+'_D'] = np.where(
        round(dataset_scaled[feature], 4) == round(dataset_scaled1[feature], 4), 0, 1
    )
print(difference.sum())

Age_D           0
SemNo_D         0
Attendance_D    0
dtype: int64
